# 04 · Tracking, person type and teams

Per analysed frame (same order as `perception.run_perception`):

1. **tracker** (`tracking.IoUTracker`: Hungarian on IoU + centre distance with a constant-velocity prediction;
   `bytetrack` optional) → track id per box, `-1` until a track is confirmed (2 hits);
2. **zone** of each person (pitch / touchline / stands, from pitch metres or the grass mask) and **person type**
   (staff, spectator, steward = hi-vis) for people off the pitch (`people.py`);
3. **kit descriptor** of the torso (every 2nd frame): chroma quartiles + hue histogram after removing this frame's
   grass colour (`teams.color_descriptor`);
4. **team / role**: with the players clicked in the setup → `KitClassifier` (nearest clicked samples per class:
   team0, team1, gk0, gk1, referee), else KMeans(2) → **vote per track** (`vote_roles`).

In [ ]:
import sys, os, time, json, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "football_eval").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
from football_eval import nbtools as nbt

P = nbt.load_params()            # notebooks/pipeline/params.yaml (or $FVE_NB_PARAMS)
ctx = nbt.build_context(P)       # config <- match setup, restricted to the window
cfg = ctx.cfg

In [ ]:
from football_eval.tracking import build_tracker
from football_eval.pitch import Pitch, draw_pitch
from football_eval.perception import foot_points, build_kit_bank
from football_eval.people import grass_top, zones, frame_types, track_types, people_summary
from football_eval.teams import TeamClassifier, KitClassifier, vote, vote_roles, CODE_NAME, torso_crop
dets = nbt.load_stage(ctx, "detections"); hom = nbt.load_stage(ctx, "homography").set_index("frame")
pitch = Pitch(cfg.pitch.length, cfg.pitch.width)
tracker = build_tracker(cfg, cfg.video.analysis_fps)
teams = TeamClassifier(cfg.teams.backend, cfg.teams.outlier_quantile, cfg.teams.outlier_factor, cfg.detection.device)
print(type(tracker).__name__, vars(tracker) if hasattr(tracker, "max_age") else "")

## 1 · The per-frame loop (tracker + zones + descriptors)

In [ ]:
rows, feats, feat_rows, crops = [], [], [], []
tm = nbt.Timer().__enter__()
for n, (f, t, img) in enumerate(nbt.frames_in_window(ctx)):
    people = nbt.detections_from_table(dets, f); tm.lap("read")
    ids = tracker.update(people); tm.lap("tracker")
    H = nbt.homography_from_row(hom.loc[f]) if f in hom.index else None
    foot, edge = foot_points(people.xyxy, img.shape)
    xy = H.to_pitch(foot) if (H is not None and len(foot)) else np.full((len(foot), 2), np.nan)
    gtop = grass_top(img)
    zone = zones(xy, foot, people.xyxy, pitch, gtop, cfg.pitch.offpitch_margin_m, cfg.pitch.staff_band_m)
    ptype = frame_types(img, people.xyxy, zone, cfg.pitch.hivis_min_share); tm.lap("zones / person type")
    desc = teams.describe(img, people.xyxy) if (n % 2 == 0 and len(people)) else [None] * len(people); tm.lap("kit descriptor")
    for k in range(len(people)):
        rows.append((f, t, int(ids[k]), people.cls[k], float(people.conf[k]), *people.xyxy[k], *foot[k], *xy[k], int(edge[k]), zone[k], ptype[k]))
        if desc[k] is not None and ptype[k] is None and ids[k] >= 0:
            feats.append(desc[k]); feat_rows.append(len(rows) - 1)
            if len(crops) < 400:
                crops.append((len(rows) - 1, torso_crop(img, people.xyxy[k]), nbt.crop(img, people.xyxy[k])))
tm.__exit__()
cols = ["frame", "t", "track_id", "cls", "conf", "x0", "y0", "x1", "y1", "u", "v", "x", "y", "edge", "zone", "ptype"]
tracks = pd.DataFrame(rows, columns=cols)
print(f"{len(tracks)} person rows, {tracks.track_id.nunique() - (tracks.track_id < 0).any()} confirmed tracks, "
      f"{(tracks.track_id < 0).mean():.1%} rows unconfirmed, {len(feats)} kit descriptors")
tm.table(n + 1)

## 2 · Track quality
Without ground truth, proxies: **fragmentation** = tracks per person actually on the pitch (1.0 ideal; > 1.5 = IDs break often), track length distribution, and ID births per frame. Short tracks are what the team vote cannot fix.

In [ ]:
tt = tracks[tracks.track_id >= 0]
life = tt.groupby("track_id").agg(first=("t", "min"), last=("t", "max"), n=("t", "size"))
life["dur_s"] = life["last"] - life["first"] + 1 / cfg.video.analysis_fps
on_pitch = tracks[tracks.zone == "pitch"].groupby("frame").size().median()
print(f"median people on the pitch per frame: {on_pitch:.0f}; tracks touching the pitch: "
      f"{tt[tt.zone == 'pitch'].track_id.nunique()} -> fragmentation {tt[tt.zone == 'pitch'].track_id.nunique() / max(on_pitch, 1):.2f}")
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
life.dur_s.hist(bins=30, ax=axes[0]); axes[0].set_title("track duration (s)")
for i, (tid, g) in enumerate(tt.groupby("track_id")):
    axes[1].plot(g.t, np.full(len(g), i), "|", ms=3)
axes[1].set_title("track timelines (one row per id)"); axes[1].set_xlabel("t (s)");

## 3 · Team / role classifier

In [ ]:
F = np.asarray(feats)
if cfg.teams.kit_samples and cfg.teams.backend == "color":
    from football_eval.detection import build_detector
    kit, seeds, kit_report = build_kit_bank(ctx.video, cfg, teams, build_detector(cfg), ctx.info)
else:
    kit, seeds, kit_report = None, {}, {}
if kit is not None:
    kit.fit(F)
    codes, margin = kit.predict(F, return_margin=True)
    print("kit classifier:", {k: v for k, v in kit.report.items() if k in ("loo_accuracy", "loo_n", "thr", "ignored_samples")})
else:
    teams.fit(F); lab = teams.predict(F)
    codes = np.where(lab >= 0, lab, -1); margin = np.full(len(codes), np.nan)
    print("unsupervised KMeans(2): team order is arbitrary (no clicked kits)")
pd.Series([CODE_NAME.get(int(c)) or "unknown" for c in codes]).value_counts().to_frame("crops")

### Crops per predicted class
Scan these: a wrong class is usually an occluded player (two shirts in one crop) or shade / motion blur.

In [ ]:
by = {}
for (ri, torso, full), c in zip(crops, codes[:len(crops)]):
    by.setdefault(CODE_NAME.get(int(c)) or "unknown", []).append(full)
for name, ims in by.items():
    ims = ims[:12]
    fig, axes = plt.subplots(1, 12, figsize=(14, 1.6))
    for ax, k in zip(axes, range(12)):
        ax.axis("off")
        if k < len(ims): ax.imshow(ims[k][..., ::-1])
    axes[0].set_title(f"{name} ({len(by[name])} crops shown max 12)", loc="left")

### Descriptor space (PCA)
Two tight, separated clouds = easy kits. Overlap = the descriptor cannot separate them (try `teams.backend: siglip`, or click more samples in different light).

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(2).fit(F)
Z = pca.transform(F)
fig, ax = plt.subplots(figsize=(7, 5))
for c in sorted(set(codes.tolist())):
    m = codes == c
    ax.scatter(Z[m, 0], Z[m, 1], s=8, alpha=0.6, label=CODE_NAME.get(int(c)) or "unknown")
if kit is not None:
    for cls_, S_ in kit.samples.items():
        zs = pca.transform(S_); ax.scatter(zs[:, 0], zs[:, 1], marker="*", s=180, edgecolor="k", label=f"clicked {cls_}")
ax.legend(fontsize=7); ax.set_title(f"kit descriptors (PCA, {pca.explained_variance_ratio_.sum():.0%} of variance)");

## 4 · Vote per track → team and role

In [ ]:
assignment = vote_roles(tracks.loc[feat_rows, "track_id"].to_numpy(), codes)
a = assignment.set_index("track_id")
tracks["team"] = tracks.track_id.map(a.team).fillna(-1).astype(int)
role = tracks.track_id.map(a.role)
sel = role.isin(["goalkeeper", "referee"])
tracks.loc[sel, "cls"] = role[sel]
tt_ = track_types(tracks)
off = tracks.track_id.map(tt_); sel = off.notna()
tracks.loc[sel, "cls"] = off[sel]; tracks.loc[sel, "team"] = -1
print(assignment.decision.value_counts().to_dict())
people_summary(tracks)

In [ ]:
f, t, img = nbt.probe_frame(ctx)
g = tracks[tracks.frame == tracks.frame.iloc[(tracks.frame - f).abs().argmin()]]
names = list(cfg.teams.team_names)
lab = [f"{r.track_id} {names[r.team][:8] if r.team >= 0 else r.cls}" for r in g.itertuples()]
col = [nbt.COLORS[int(r.team)] if r.team >= 0 else (nbt.COLORS["referee"] if r.cls == "referee" else (160, 160, 160)) for r in g.itertuples()]
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1.6, 1]})
nbt.show(nbt.draw_boxes(img, g[["x0", "y0", "x1", "y1"]].to_numpy(), lab, col), "track id + team", ax=axes[0])
draw_pitch(axes[1], pitch)
for tm_, c in ((0, "C3"), (1, "C0"), (-1, "0.5")):
    q = g[g.team == tm_]; axes[1].scatter(q.x, q.y, c=c, s=30, label=names[tm_] if tm_ >= 0 else "other")
axes[1].legend(fontsize=7);

In [ ]:
nbt.save_stage(ctx, "tracks", tracks); nbt.save_stage(ctx, "team_assignment", assignment);